## Data Cleaning

Look at newsroom_data_exploration.ipynb to explore the initial dataset and understand changes

In [13]:
import pandas as pd
import csv
from collections import Counter

file_path = "newsroom_cleaned.csv"
output_file = "newsroom_final_cleaned.csv.gz"
chunk_size = 200000

# DOMAIN CLEANING
def clean_domain(domain):
    if pd.isna(domain):
        return None

    domain = str(domain).lower().strip()
    domain = domain.replace("www.", "")

    # fix abcnews issue
    if "abcnews" in domain:
        return "abcnews.com"

    parts = domain.split(".")

    if len(parts) >= 3 and parts[-2] in ["co", "com", "org", "net"]:
        return ".".join(parts[-3:])

    if len(parts) >= 2:
        return ".".join(parts[-2:])

    return domain

# DATE PARSER
def parse_date(date_val):
    try:
        if pd.isna(date_val):
            return pd.NaT

        date_str = str(date_val).strip()

        if date_str == "" or date_str.lower() in ["nan", "none", "nat"]:
            return pd.NaT

        if len(date_str) == 14:
            return pd.to_datetime(date_str, format="%Y%m%d%H%M%S", errors="coerce")
        elif len(date_str) == 10:
            return pd.to_datetime(date_str, format="%Y%m%d%H", errors="coerce")
        else:
            return pd.NaT
    except:
        return pd.NaT

# FIRST PASS TO GET VALID PUBLISHERS
publisher_counter = Counter()

for chunk in pd.read_csv(file_path, usecols=["domain"], chunksize=chunk_size):
    clean_domains = chunk["domain"].apply(clean_domain)
    publisher_counter.update(clean_domains.dropna())

publisher_counts = pd.Series(publisher_counter).sort_values(ascending=False)
valid_publishers = set(publisher_counts[publisher_counts >= 5000].index)

print("Publishers kept:", len(valid_publishers))
print(publisher_counts[publisher_counts >= 5000])

Publishers kept: 30
nytimes.com           186095
washingtonpost.com    117074
foxnews.com            95619
theguardian.com        70706
nydailynews.com        67614
wsj.com                60879
usatoday.com           54330
cnn.com                52649
time.com               51663
mashable.com           38626
people.com             37423
forbes.com             36645
fortune.com            28093
cnbc.com               26738
foxsports.com          26724
telegraph.co.uk        25477
9news.com.au           24599
tmz.com                23118
latimes.com            22334
bostonglobe.com        20754
sfgate.com             18090
thesun.co.uk           17105
cbc.ca                 16375
aol.com                15176
nypost.com             12687
abcnews.com            12023
bbc.com                12001
nbcnews.com             9413
reuters.com             7223
bloomberg.com           6959
dtype: int64


In [14]:
# SECOND PASS TO FULL CLEANING
first_chunk = True
seen_urls = set()

rows_in = 0
rows_out = 0
missing_url_removed = 0
missing_date_removed = 0
dup_removed = 0
outliers_removed = 0

for chunk in pd.read_csv(
    file_path,
    chunksize=chunk_size,
    dtype={"date": str},
    keep_default_na=True,
    na_values=["", "nan", "NaN", "None", "NA", "N/A"]
):

    rows_in += len(chunk)

    # CLEAN TEXT COLUMNS
    for col in ["title", "summary", "text"]:
        if col in chunk.columns:
            chunk[col] = chunk[col].fillna("").astype(str).str.strip()

    # CLEAN URL / DOMAIN
    for col in ["url", "domain"]:
        if col in chunk.columns:
            chunk[col] = chunk[col].astype("string").str.strip()

    # NORMALIZE DOMAIN
    chunk["publisher_clean"] = chunk["domain"].apply(clean_domain)

    # FILTER VALID PUBLISHERS
    chunk = chunk[chunk["publisher_clean"].isin(valid_publishers)].copy()

    # REMOVE MISSING URL
    before = len(chunk)
    chunk = chunk[
        chunk["url"].notna() &
        (~chunk["url"].isin(["", "nan", "NaN", "None"]))
    ].copy()
    missing_url_removed += (before - len(chunk))

    # REMOVE DUPLICATES (URL)
    before = len(chunk)
    is_new = ~chunk["url"].isin(seen_urls)
    chunk = chunk[is_new].copy()
    seen_urls.update(chunk["url"].tolist())
    dup_removed += (before - len(chunk))

    # PARSE DATE
    chunk["datetime"] = chunk["date"].apply(parse_date)

    # REMOVE MISSING / INVALID DATE
    before = len(chunk)
    chunk = chunk[chunk["datetime"].notna()].copy()
    missing_date_removed += (before - len(chunk))

    # CREATE TIME FEATURES
    chunk["year"] = chunk["datetime"].dt.year.astype("Int64")
    chunk["month"] = chunk["datetime"].dt.month.astype("Int64")
    chunk["year_month"] = chunk["datetime"].dt.to_period("M").astype(str)

    # LENGTH FEATURES
    chunk["text_len"] = chunk["text"].str.len()
    chunk["summary_len"] = chunk["summary"].str.len()
    chunk["title_len"] = chunk["title"].str.len()

    # REMOVE EMPTY TITLES
    chunk = chunk[chunk["title_len"] > 0].copy()

    # REMOVE OUTLIERS
    before = len(chunk)
    chunk = chunk[
        (chunk["text_len"] < 50000) &
        (chunk["summary_len"] < 2000)
    ].copy()
    outliers_removed += (before - len(chunk))

    rows_out += len(chunk)

    # SAVE FINAL OUTPUT
    if first_chunk:
        chunk.to_csv(
            output_file,
            index=False,
            mode="w",
            quoting=csv.QUOTE_ALL,
            compression="gzip"
        )
        first_chunk = False
    else:
        chunk.to_csv(
            output_file,
            index=False,
            mode="a",
            header=False,
            quoting=csv.QUOTE_ALL,
            compression="gzip"
        )

print("\nFINAL DATASET CREATED")
print("Rows read:", rows_in)
print("Rows kept:", rows_out)
print("Missing URL removed:", missing_url_removed)
print("Missing date removed:", missing_date_removed)
print("Duplicates removed:", dup_removed)
print("Outliers removed:", outliers_removed)


FINAL DATASET CREATED
Rows read: 1212740
Rows kept: 1190870
Missing URL removed: 0
Missing date removed: 0
Duplicates removed: 760
Outliers removed: 2497


## More Data Exploring

In [3]:
import pandas as pd

df = pd.read_csv("newsroom_final_cleaned.csv.gz")

In [21]:
print("Rows:", len(df))
print("Publishers:", df["publisher_clean"].nunique())

print("\nPublishers:")
print(df["publisher_clean"].value_counts().head(40))

print("\nDuplicate URLs:", df["url"].duplicated().sum())
print("Missing datetime:", df["datetime"].isna().sum())

print("\nYear distribution:")
print(df["year"].value_counts().sort_index())

print("\nLength summaries:")
print(df[["text_len", "summary_len", "title_len"]].describe())

Rows: 1190870
Publishers: 30

Publishers:
nytimes.com           185309
washingtonpost.com    116238
foxnews.com            95504
theguardian.com        70697
nydailynews.com        67604
wsj.com                60870
usatoday.com           54318
cnn.com                52488
time.com               51642
mashable.com           38609
people.com             37412
forbes.com             36640
fortune.com            28037
cnbc.com               26738
foxsports.com          26705
telegraph.co.uk        25461
9news.com.au           24598
tmz.com                23118
latimes.com            22322
bostonglobe.com        20740
thesun.co.uk           17099
sfgate.com             16930
cbc.ca                 16374
aol.com                15144
nypost.com             12687
abcnews.com            12013
bbc.com                11999
nbcnews.com             9401
reuters.com             7219
bloomberg.com           6954
Name: publisher_clean, dtype: int64

Duplicate URLs: 130
Missing datetime: 0

Year distr

In [22]:
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)


Columns:
['text', 'summary', 'title', 'url', 'date', 'density_bin', 'coverage_bin', 'compression_bin', 'density', 'coverage', 'compression', 'domain', 'publisher_clean', 'datetime', 'year', 'month', 'year_month', 'text_len', 'summary_len', 'title_len']

Data types:
text                object
summary             object
title               object
url                 object
date                 int64
density_bin         object
coverage_bin        object
compression_bin     object
density            float64
coverage           float64
compression        float64
domain              object
publisher_clean     object
datetime            object
year                 int64
month                int64
year_month          object
text_len             int64
summary_len          int64
title_len            int64
dtype: object


In [23]:
missing = df.isna().mean().sort_values(ascending=False)
print("\nMissingness (%):")
print(missing)


Missingness (%):
summary            0.000005
text               0.000000
summary_len        0.000000
text_len           0.000000
year_month         0.000000
month              0.000000
year               0.000000
datetime           0.000000
publisher_clean    0.000000
domain             0.000000
compression        0.000000
coverage           0.000000
density            0.000000
compression_bin    0.000000
coverage_bin       0.000000
density_bin        0.000000
date               0.000000
url                0.000000
title              0.000000
title_len          0.000000
dtype: float64


In [24]:
df.head(10)

,text,summary,title,url,date,density_bin,coverage_bin,compression_bin,density,coverage,compression,domain,publisher_clean,datetime,year,month,year_month,text_len,summary_len,title_len
0,"HAMBURG, Germany, June 3  As he left the socc...",A surge in discriminatory behavior toward blac...,Surge in Racist Mood Raises Concerns on Eve of...,http://www.nytimes.com/2006/06/04/sports/socce...,20060618204254,mixed,high,high,7.823529,1.000000,137.470580,nytimes.com,nytimes.com,2006-06-18 20:42:54,2006,6,2006-06,12028,105,56
1,"WASHINGTON, Dec. 23 - The National Security Ag...","The volume of information harvested, without \...","Spy Agency Mined Vast Data Trove, Officials Re...",http://www.nytimes.com/2005/12/24/politics/24s...,20060620043011,mixed,medium,medium,4.727272,0.909091,33.636364,nytimes.com,nytimes.com,2006-06-20 04:30:11,2006,6,2006-06,4282,118,50
2,IF outsized executive pay has indeed become a ...,The battle between Pfizer Inc.'s investors and...,Investors vs. Pfizer: Guess Who Has the Guns?,http://www.nytimes.com/2006/04/23/business/you...,20060909062911,extractive,high,medium,11.720000,1.000000,33.880000,nytimes.com,nytimes.com,2006-09-09 06:29:11,2006,9,2006-09,4654,151,45
3,BY A.J. BENZA & MICHAEL LEWITTES\n\nIf Simon R...,"If Simon Rex looks a little familiar, it may n...",REX FLEXED PECS FOR SKIN PICS,http://www.nydailynews.com/archives/gossip/199...,20080313232743,extractive,high,low,38.988235,0.988235,11.894117,nydailynews.com,nydailynews.com,2008-03-13 23:27:43,2008,3,2008-03,4553,370,29
4,Spinach has terrorized generations of veggie-p...,POPEYE-WORTHY PIE. PHYLLO DOUGH WRAPS SPINACH ...,POPEYE-WORTHY PIE. PHYLLO DOUGH WRAPS SPINACH ...,http://www.nydailynews.com/archives/entertainm...,20080314003027,extractive,medium,low,36.629215,0.921348,3.932584,nydailynews.com,nydailynews.com,2008-03-14 00:30:27,2008,3,2008-03,1649,454,58
5,"All day, every day, Cheryl Bernstein thanks he...","All day, every day, Cheryl Bernstein thanks he...",JOY FOR ADDICTS ON MEND AS CHILDREN ARE RETURNED,http://www.nydailynews.com/archives/news/2001/...,20080520122148,extractive,high,low,23.475609,0.987805,4.597561,nydailynews.com,nydailynews.com,2008-05-20 12:21:48,2008,5,2008-05,1841,402,48
6,With Police Commissioner Bernard Kerik crackin...,By JOHN MARZULLI DAILY NEWS POLICE BUREAU CHIE...,QUICK WORK BY THE COPS NYPD response time plunges,http://www.nydailynews.com/archives/news/2001/...,20080711053245,extractive,medium,medium,16.890244,0.939024,20.085365,nydailynews.com,nydailynews.com,2008-07-11 05:32:45,2008,7,2008-07,7945,437,49
7,"Wednesday, April 19th 1995, 2:35AM\n\nJail inm...",Jail inmates flout the city's newest law every...,JAILED SMOKERS PUT CIG LAW TO TEST,http://www.nydailynews.com/archives/news/1995/...,20090205180750,extractive,high,low,22.234568,0.975309,4.382716,nydailynews.com,nydailynews.com,2009-02-05 18:07:50,2009,2,2009-02,1619,402,34
8,BY GEORGE RUSH AND JOANNA MOLLOY With Kasia An...,Did Tatum O'Neal's latest battle with ex-husba...,REPORT: TATUM FIGHTS ANOTHER ROUND,http://www.nydailynews.com/archives/gossip/200...,20090208012929,extractive,high,medium,17.723684,0.986842,18.868422,nydailynews.com,nydailynews.com,2009-02-08 01:29:29,2009,2,2009-02,6879,402,34
9,"Wednesday, May 18th 2005, 9:59AM\n\nSummer is ...",COOL COCKTAIL A summer drink you'll cotton to ...,A SUMMER DRINK YOU'LL COTTON TO,http://www.nydailynews.com/archives/lifestyle/...,20090310185130,extractive,medium,low,42.658825,0.941176,1.752941,nydailynews.com,nydailynews.com,2009-03-10 18:51:30,2009,3,2009-03,769,469,31


## More Data Cleaning

In [4]:
# KEEP ONLY 2005 OR NEWER
df = df[df["year"] >= 2005].copy()

# ADD DAY + YYYY-MM-DD COLUMN
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["day"] = df["datetime"].dt.day.astype("Int64")
df["year_month_day"] = df["datetime"].dt.strftime("%Y-%m-%d")

# CLEAN PUBLISHER LABEL
def simplify_publisher(pub):
    if pd.isna(pub):
        return pub
    
    pub = str(pub).lower().strip()
    
    if pub.endswith(".co.uk"):
        pub = pub[:-6]
    elif pub.endswith(".com"):
        pub = pub[:-4]
    elif pub.endswith(".ca"):
        pub = pub[:-3]
    elif pub.endswith(".au"):
        pub = pub[:-3]
    elif pub.endswith(".org"):
        pub = pub[:-4]
    elif pub.endswith(".net"):
        pub = pub[:-4]
    
    return pub

df["publisher_clean"] = df["publisher_clean"].apply(simplify_publisher)

# DROP UNNECESSARY COLUMNS
cols_to_drop = [
    "density_bin", "coverage_bin", "compression_bin",
    "density", "coverage", "compression",
    "year_month"
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# REORDER COLUMNS
preferred_order = [
    "title", "summary", "text", "url",
    "domain", "publisher_clean",
    "date", "datetime",
    "year", "month", "day", "year_month_day",
    "title_len", "summary_len", "text_len"
]

remaining_cols = [c for c in df.columns if c not in preferred_order]
df = df[[c for c in preferred_order if c in df.columns] + remaining_cols]

In [8]:
# SAVE
import csv

output_file = "newsroom_analysis_ready.csv.gz"

df.to_csv(
    output_file,
    index=False,
    compression="gzip",
    quoting=csv.QUOTE_ALL
)

print("Saved:", output_file)

Saved: newsroom_analysis_ready.csv.gz


## FINAL DATASET

In [9]:
import pandas as pd

df = pd.read_csv("newsroom_analysis_ready.csv.gz")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nPublishers:")
print(df["publisher_clean"].value_counts().head(50))

print("\nYear distribution:")
print(df["year"].value_counts().sort_index())

Shape: (1189862, 15)

Columns:
['title', 'summary', 'text', 'url', 'domain', 'publisher_clean', 'date', 'datetime', 'year', 'month', 'day', 'year_month_day', 'title_len', 'summary_len', 'text_len']

Publishers:
nytimes           185251
washingtonpost    116238
foxnews            95504
theguardian        70697
nydailynews        67604
wsj                60854
usatoday           54315
cnn                52315
time               51642
mashable           38609
people             37412
forbes             36636
fortune            28037
cnbc               26738
foxsports          26649
telegraph          25424
9news.com          24598
tmz                23118
latimes            22239
bostonglobe        20740
thesun             17099
sfgate             16930
cbc                16210
aol                15143
nypost             12635
bbc                11999
abcnews            11656
nbcnews             9401
reuters             7215
bloomberg           6954
Name: publisher_clean, dtype: int64

Ye

In [10]:
df.head(15)

,title,summary,text,url,domain,publisher_clean,date,datetime,year,month,day,year_month_day,title_len,summary_len,text_len
0,Surge in Racist Mood Raises Concerns on Eve of...,A surge in discriminatory behavior toward blac...,"HAMBURG, Germany, June 3  As he left the socc...",http://www.nytimes.com/2006/06/04/sports/socce...,nytimes.com,nytimes,20060618204254,2006-06-18 20:42:54,2006,6,18,2006-06-18,56,105,12028
1,"Spy Agency Mined Vast Data Trove, Officials Re...","The volume of information harvested, without \...","WASHINGTON, Dec. 23 - The National Security Ag...",http://www.nytimes.com/2005/12/24/politics/24s...,nytimes.com,nytimes,20060620043011,2006-06-20 04:30:11,2006,6,20,2006-06-20,50,118,4282
2,Investors vs. Pfizer: Guess Who Has the Guns?,The battle between Pfizer Inc.'s investors and...,IF outsized executive pay has indeed become a ...,http://www.nytimes.com/2006/04/23/business/you...,nytimes.com,nytimes,20060909062911,2006-09-09 06:29:11,2006,9,9,2006-09-09,45,151,4654
3,REX FLEXED PECS FOR SKIN PICS,"If Simon Rex looks a little familiar, it may n...",BY A.J. BENZA & MICHAEL LEWITTES\n\nIf Simon R...,http://www.nydailynews.com/archives/gossip/199...,nydailynews.com,nydailynews,20080313232743,2008-03-13 23:27:43,2008,3,13,2008-03-13,29,370,4553
4,POPEYE-WORTHY PIE. PHYLLO DOUGH WRAPS SPINACH ...,POPEYE-WORTHY PIE. PHYLLO DOUGH WRAPS SPINACH ...,Spinach has terrorized generations of veggie-p...,http://www.nydailynews.com/archives/entertainm...,nydailynews.com,nydailynews,20080314003027,2008-03-14 00:30:27,2008,3,14,2008-03-14,58,454,1649
5,JOY FOR ADDICTS ON MEND AS CHILDREN ARE RETURNED,"All day, every day, Cheryl Bernstein thanks he...","All day, every day, Cheryl Bernstein thanks he...",http://www.nydailynews.com/archives/news/2001/...,nydailynews.com,nydailynews,20080520122148,2008-05-20 12:21:48,2008,5,20,2008-05-20,48,402,1841
6,QUICK WORK BY THE COPS NYPD response time plunges,By JOHN MARZULLI DAILY NEWS POLICE BUREAU CHIE...,With Police Commissioner Bernard Kerik crackin...,http://www.nydailynews.com/archives/news/2001/...,nydailynews.com,nydailynews,20080711053245,2008-07-11 05:32:45,2008,7,11,2008-07-11,49,437,7945
7,JAILED SMOKERS PUT CIG LAW TO TEST,Jail inmates flout the city's newest law every...,"Wednesday, April 19th 1995, 2:35AM\n\nJail inm...",http://www.nydailynews.com/archives/news/1995/...,nydailynews.com,nydailynews,20090205180750,2009-02-05 18:07:50,2009,2,5,2009-02-05,34,402,1619
8,REPORT: TATUM FIGHTS ANOTHER ROUND,Did Tatum O'Neal's latest battle with ex-husba...,BY GEORGE RUSH AND JOANNA MOLLOY With Kasia An...,http://www.nydailynews.com/archives/gossip/200...,nydailynews.com,nydailynews,20090208012929,2009-02-08 01:29:29,2009,2,8,2009-02-08,34,402,6879
9,A SUMMER DRINK YOU'LL COTTON TO,COOL COCKTAIL A summer drink you'll cotton to ...,"Wednesday, May 18th 2005, 9:59AM\n\nSummer is ...",http://www.nydailynews.com/archives/lifestyle/...,nydailynews.com,nydailynews,20090310185130,2009-03-10 18:51:30,2009,3,10,2009-03-10,31,469,769
